In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)

from peft import PeftModel
import torch 

In [ ]:
checkpoint = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    quantization_config=bnb,
    device_map="auto")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "finance_chat_model"
)  

In [ ]:
model = PeftModel.from_pretrained(
    base_model,
    "finance_chat_model"
) 

In [ ]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
) 

In [ ]:
fine_tuned_generator=pipeline("text-generation",model=model,tokenizer=tokenizer)
base_generator=pipeline("text-generation",model=checkpoint,tokenizer=tokenizer)  

In [ ]:
with open("model_comparison.txt", "w", encoding="utf-8") as f:

    questions = [
        "What is asset allocation in a portfolio?.",
        "How can rising interest rates affect bond prices?",
        "Explain the benefits of investing through SIPs",
        "A person wants passive income from investments. What options exist?",
        "Compare SIP and lump sum investing for a beginner."]
        

    for q in questions:

        # prompt = f"<s>[INST] {q} [/INST]"
        prompt = f"Question: {q}\nAnswer:"

        print("\n" + "=" * 80)
        print("QUESTION:", q)

        # Base Model
        base_result = base_generator(
            prompt,
            max_new_tokens=150,
            repetition_penalty=1.2,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            return_full_text=False
        )

        # Fine-Tuned Model
        fine_tuned_result = fine_tuned_generator(
            prompt,
            max_new_tokens=150,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            return_full_text=False
        )

        print("\nBASE MODEL:")
        print(base_result[0]["generated_text"])

        print("\nFINE-TUNED MODEL:")
        print(fine_tuned_result[0]["generated_text"])

        # Save to file
        f.write("\n" + "=" * 80 + "\n")
        f.write(f"QUESTION: {q}\n\n")

        f.write("BASE MODEL:\n")
        f.write(base_result[0]["generated_text"])

        f.write("\n\nFINE-TUNED MODEL:\n")
        f.write(fine_tuned_result[0]["generated_text"])

        f.write("\n\n")